# VQ-VAE 编码与重建效果评估

本 Notebook 用于：
- 加载训练好的 VQ-VAE 模型
- 在指定数据集 split 上做重建评估
- 统计重建指标（MSE/MAE/RMSE/PSNR/R2）
- 统计量化器指标（Perplexity/Codebook Usage/VQ Loss）
- 可视化原场、重建场与误差

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['image.cmap'] = 'viridis'

# 项目根目录（当前 notebook 在 script/notebook）
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from model.vqvae import VQVAE
from data.dataset import build_dataset, load_constants
from data.data_utils import normalize_fn
from config import get_dataset_config

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DEVICE:', DEVICE)

In [ ]:
# ====== 可修改配置 ======
DATA_NAME = 'glorys12_kuroshio_extension'
TAG = 'vqvae_hd64128256_ed128_ne2048_ema'
SPLIT = 'val'  # train / val / test
BATCH_SIZE = 4
NUM_WORKERS = 4
MAX_EVAL_BATCHES = 10  # None 表示全部

# VQ-VAE 结构（需与训练一致）
HIDDEN_DIMS = [64, 128, 256]
EMBEDDING_DIM = 128
NUM_EMBEDDINGS = 2048
QUANTIZER = 'ema'  # ema / standard
EMA_DECAY = 0.99
COMMITMENT_COST = 0.25

# checkpoint 路径
CKPT_PATH = os.path.join(PROJECT_ROOT, 'output', DATA_NAME, TAG, 'best_model.pth')

assert os.path.exists(CKPT_PATH), f'checkpoint not found: {CKPT_PATH}'
print('CKPT_PATH:', CKPT_PATH)

In [ ]:
dataset_config = get_dataset_config(DATA_NAME)

_date_ranges = {
    'train': dataset_config.train_date_range,
    'val':   dataset_config.val_date_range,
    'test':  dataset_config.test_date_range,
}
dataset = build_dataset(dataset_config.raw_data_dir, _date_ranges[SPLIT])
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

constants = load_constants(dataset_config.constant_dir)
normed_ocean_mean = constants[0].to(DEVICE)
normed_ocean_std = constants[1].to(DEVICE)
mask = constants[-1].to(DEVICE).float()

normalization_stats = {
    'ocean': {
        'mu': normed_ocean_mean[..., None, None],
        'sigma': normed_ocean_std[..., None, None],
    }
}

print('Dataset split:', SPLIT, 'size:', len(dataset))
print('Mask shape:', tuple(mask.shape))
print('Num channels:', dataset_config.num_channels)

In [ ]:
def load_model_weights(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict):
        state_dict = checkpoint.get('model_state_dict', checkpoint.get('model', checkpoint.get('net', checkpoint)))
    else:
        state_dict = checkpoint

    normalized_state = {}
    for key, value in state_dict.items():
        if key.startswith('module.'):
            normalized_state[key[7:]] = value
        else:
            normalized_state[key] = value

    model.load_state_dict(normalized_state, strict=True)

in_channels = dataset_config.num_channels
model = VQVAE(
    in_channels=in_channels,
    hidden_dims=HIDDEN_DIMS,
    embedding_dim=EMBEDDING_DIM,
    num_embeddings=NUM_EMBEDDINGS,
    commitment_cost=COMMITMENT_COST,
    quantizer=QUANTIZER,
    ema_decay=EMA_DECAY,
).to(DEVICE)

load_model_weights(model, CKPT_PATH, DEVICE)
model.eval()
print(f'Model loaded.  Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def preprocess_raw_data(batch, normalization_stats, device):
    x = batch['raw_data'].to(device=device) if isinstance(batch, dict) else batch.to(device=device)
    x = normalize_fn(x, **normalization_stats['ocean'])
    return x.to(dtype=torch.float32)

def codebook_usage_ratio(indices, num_embeddings):
    unique_codes = torch.unique(indices).numel()
    return float(unique_codes) / float(num_embeddings)

def masked_metrics(pred, target, mask, eps=1e-8):
    """per-sample 指标，与 eval_dcae 保持一致。返回长度为 B 的 list of dict。"""
    m = mask.to(pred.device)
    while m.ndim < pred.ndim:
        m = m.unsqueeze(0)
    m = m.expand_as(pred)

    B    = pred.shape[0]
    flat = lambda t: t.reshape(B, -1)

    n_valid = flat(m).sum(1).clamp_min(eps)
    diff    = (pred - target) * m
    sq_err  = flat(diff.pow(2)).sum(1)
    abs_err = flat(diff.abs()).sum(1)
    mse     = sq_err / n_valid
    mae     = abs_err / n_valid
    rmse    = mse.sqrt()

    mean_t = flat(target * m).sum(1) / n_valid
    ss_tot = flat((target - mean_t.view(B, 1, 1, 1)).pow(2) * m).sum(1).clamp_min(eps)
    r2     = 1.0 - sq_err / ss_tot

    INF    = 1e10
    oc_max = (target * m + (1 - m) * (-INF)).reshape(B, -1).max(1).values
    oc_min = (target * m + (1 - m) *   INF ).reshape(B, -1).min(1).values
    psnr   = 20.0 * torch.log10((oc_max - oc_min).clamp_min(eps) / rmse.clamp_min(eps))

    return [
        {'mse': mse[i].item(), 'mae': mae[i].item(),
         'rmse': rmse[i].item(), 'r2': r2[i].item(), 'psnr': psnr[i].item()}
        for i in range(B)
    ]

@torch.no_grad()
def run_evaluation(model, loader, mask, normalization_stats, device, num_embeddings, max_batches=None):
    records          = []
    all_perplexity   = []
    all_usage        = []
    first_batch_cache = None

    for step, batch in enumerate(loader):
        if max_batches is not None and step >= max_batches:
            break

        x = preprocess_raw_data(batch, normalization_stats, device)
        recon, q_info = model(x)

        vq_loss        = float(q_info['vq_loss'].item())
        codebook_loss  = float(q_info['codebook_loss'].item())
        commitment_loss = float(q_info['commitment_loss'].item())
        perplexity     = float(q_info['perplexity'].item())
        usage_ratio    = codebook_usage_ratio(q_info['indices'], num_embeddings)

        for met in masked_metrics(recon, x, mask, eps=1e-8):
            met['step']            = step
            met['vq_loss']         = vq_loss
            met['codebook_loss']   = codebook_loss
            met['commitment_loss'] = commitment_loss
            met['perplexity']      = perplexity
            met['usage_ratio']     = usage_ratio
            records.append(met)

        all_perplexity.append(perplexity)
        all_usage.append(usage_ratio)

        if first_batch_cache is None:
            first_batch_cache = {
                'x':       x.detach().cpu(),
                'recon':   recon.detach().cpu(),
                'indices': q_info['indices'].detach().cpu(),
            }

    df = pd.DataFrame(records)
    summary_cols = ['mse', 'mae', 'rmse', 'r2', 'psnr',
                    'vq_loss', 'codebook_loss', 'commitment_loss', 'perplexity', 'usage_ratio']
    summary = df[summary_cols].mean().to_dict() if len(df) > 0 else {}

    extra_stats = {
        'perplexity_mean':   float(np.mean(all_perplexity)) if all_perplexity else np.nan,
        'perplexity_std':    float(np.std(all_perplexity))  if all_perplexity else np.nan,
        'usage_ratio_mean':  float(np.mean(all_usage))      if all_usage      else np.nan,
        'usage_ratio_std':   float(np.std(all_usage))       if all_usage      else np.nan,
    }

    return df, summary, extra_stats, first_batch_cache

df_metrics, summary_metrics, q_stats, vis_cache = run_evaluation(
    model=model,
    loader=loader,
    mask=mask,
    normalization_stats=normalization_stats,
    device=DEVICE,
    num_embeddings=NUM_EMBEDDINGS,
    max_batches=MAX_EVAL_BATCHES,
)

print(f'Evaluated {len(df_metrics)} samples\n')
print('Summary metrics:')
for k, v in summary_metrics.items():
    print(f'  {k}: {v:.6f}')
print('Quantizer stats:', q_stats)
df_metrics.head(10)

In [ ]:
# 指标分布可视化
if len(df_metrics) > 0:
    cols = ['mse', 'mae', 'rmse', 'r2', 'psnr', 'perplexity', 'usage_ratio', 'vq_loss']
    fig, axes = plt.subplots(2, 4, figsize=(20, 6))
    axes = axes.ravel()
    for i, c in enumerate(cols):
        axes[i].hist(df_metrics[c].values, bins=20)
        axes[i].set_title(c)
    plt.tight_layout()
    plt.show()
else:
    print('No evaluation records.')

In [ ]:
# 重建可视化：原场/重建/绝对误差（反归一化）
assert vis_cache is not None, 'No cached batch for visualization.'

x = vis_cache['x']
recon = vis_cache['recon']

# 反归一化到物理量空间
mu = normalization_stats['ocean']['mu'].detach().cpu()
sigma = normalization_stats['ocean']['sigma'].detach().cpu()
x = x * sigma + mu
recon = recon * sigma + mu

SAMPLE_ID = 0
max_channel = x.shape[1] - 1
CHANNELS = sorted(set([0, max_channel // 4, max_channel // 2, 3 * max_channel // 4, max_channel]))

# 掩膜放到 CPU，并广播到 batch
m = mask.detach().cpu()
while m.ndim < x.ndim:
    m = m.unsqueeze(0)
if m.shape[0] == 1 and x.shape[0] > 1:
    m = m.expand(x.shape[0], *m.shape[1:])

rows = len(CHANNELS)
fig, axes = plt.subplots(rows, 3, figsize=(12, 3 * rows))
if rows == 1:
    axes = np.expand_dims(axes, axis=0)

for r, ch in enumerate(CHANNELS):
    gt = x[SAMPLE_ID, ch].numpy()
    rc = recon[SAMPLE_ID, ch].numpy()
    mk = m[SAMPLE_ID, ch].numpy().astype(bool)

    gt = np.where(mk, gt, np.nan)
    rc = np.where(mk, rc, np.nan)
    er = np.abs(rc - gt)

    shared_min = np.nanmin(np.stack([gt, rc]))
    shared_max = np.nanmax(np.stack([gt, rc]))
    err_max = np.nanpercentile(er, 99) if np.isfinite(er).any() else 1.0
    err_max = max(float(err_max), 1e-8)

    im0 = axes[r, 0].imshow(gt, vmin=shared_min, vmax=shared_max)
    axes[r, 0].set_title(f'CH{ch} | Input(denorm)')
    plt.colorbar(im0, ax=axes[r, 0], fraction=0.046, pad=0.04)

    im1 = axes[r, 1].imshow(rc, vmin=shared_min, vmax=shared_max)
    axes[r, 1].set_title(f'CH{ch} | Recon(denorm)')
    plt.colorbar(im1, ax=axes[r, 1], fraction=0.046, pad=0.04)

    im2 = axes[r, 2].imshow(er, vmin=0.0, vmax=err_max)
    axes[r, 2].set_title(f'CH{ch} | Abs Error(denorm)')
    plt.colorbar(im2, ax=axes[r, 2], fraction=0.046, pad=0.04)

for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# 码本索引可视化（首个样本）
indices = vis_cache['indices']
idx_map = indices[0].numpy()

plt.figure(figsize=(6, 5))
plt.imshow(idx_map)
plt.title('Codebook Indices Map (sample 0)')
plt.colorbar(fraction=0.046, pad=0.04)
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

print('Unique codes in sample 0:', np.unique(idx_map).size)
print('Usage ratio in sample 0:', np.unique(idx_map).size / NUM_EMBEDDINGS)